# 🎯 Inferencia con el mejor modelo (final, sobre `src`)

> Capa **final**: carga `models/best_model.pkl` (XGBoost, el ganador) y predice reservas nuevas, igual que hacen la API y la UI. Glosario: [`../docs/glosario.md`](../docs/glosario.md).

Este cuaderno es la **última capa** del proyecto: ya no entrena ni compara nada, solo **usa** el modelo ganador para predecir. La gracia es que importa exactamente las mismas funciones que el resto del sistema (`ml_hotel_cancellations.ml.predict`), de modo que lo que ves aquí es el mismo cálculo que devuelve la API en producción.

Recorrido:

1. **Cargar el modelo** persistido (`best_model.pkl`).
2. **Predecir una reserva de ejemplo** (la `BOOKING_EXAMPLE` del contrato).
3. **Predecir un lote** de reservas reales del dataset y comparar aciertos.
4. *(Opcional)* **Llamar a la API** para ver que sirve el mismo modelo.
5. **Conclusión**: una sola fuente de verdad para la inferencia.

## ⚙️ Configuración e imports

Importamos el contrato y las funciones de inferencia desde `src`. No reimplementamos nada: `load_best_model`, `prepare_for_inference` y `predict_dataframe` son las mismas que usan la API y la CLI `predict`.

In [1]:
%matplotlib inline
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
from ml_hotel_cancellations import config
from ml_hotel_cancellations.ml.predict import (
    load_best_model,
    prepare_for_inference,
    predict_dataframe,
)
from ml_hotel_cancellations.ml.data_loader import load_raw_data

print('Modelo:', config.BEST_MODEL_PATH)
print('Umbral de decisión:', config.DECISION_THRESHOLD)
print('Columna objetivo:', config.TARGET_COLUMN)

Modelo: /home/manu/dev/github.com/manupm87/pontia-ml/models/best_model.pkl
Umbral de decisión: 0.5
Columna objetivo: is_canceled


## 1. Cargar el modelo

`load_best_model()` deserializa `models/best_model.pkl` con `joblib`. Lo que se guardó durante el entrenamiento **no** es solo el clasificador: es un `Pipeline` completo de scikit-learn con dos pasos encadenados:

1. el **preprocesador** (imputación, escalado y *one-hot encoding*), y
2. el **clasificador XGBoost** ya ajustado.

Por eso al modelo se le pasan las columnas **en crudo**: él aplica por dentro el mismo preprocesado que vio en entrenamiento, sin que tengamos que repetirlo. Esto elimina el clásico *training/serving skew* (que el preprocesado de inferencia difiera del de entrenamiento).

In [2]:
modelo = load_best_model()
print('Tipo del objeto cargado:', type(modelo).__name__)
print()
# Si es un Pipeline, mostramos sus pasos (preprocesador + clasificador).
if hasattr(modelo, 'named_steps'):
    for nombre, paso in modelo.named_steps.items():
        print(f'  - {nombre}: {type(paso).__name__}')
else:
    print('  (objeto no-Pipeline:', type(modelo).__name__, ')')

Tipo del objeto cargado: Pipeline

  - preprocessor: ColumnTransformer
  - model: XGBClassifier


## 2. Predecir una reserva de ejemplo

`config.BOOKING_EXAMPLE` es la reserva de ejemplo del **contrato** (las 27 características con los nombres exactos que espera el `Pipeline`). Es la misma que usan el esquema Pydantic de la API y el formulario de la UI. La convertimos en un DataFrame de **una fila** y la pasamos por `predict_dataframe`.

In [3]:
df_ejemplo = pd.DataFrame([config.BOOKING_EXAMPLE])
print('Forma:', df_ejemplo.shape, '(1 reserva x 27 features)')
df_ejemplo.T  # transpuesta para verla cómoda en vertical

Forma: (1, 27) (1 reserva x 27 features)


,0
hotel,City Hotel
lead_time,100
arrival_date_month,August
arrival_date_week_number,33
arrival_date_day_of_month,15
stays_in_weekend_nights,2
stays_in_week_nights,5
adults,2
children,0
babies,0


`predict_dataframe` devuelve un DataFrame con dos columnas:

- **`prediction`**: la clase 0/1 (0 = no cancela, 1 = cancela).
- **`probability_canceled`**: la probabilidad estimada de cancelación, P(cancela).

La clase sale de aplicar el umbral `config.DECISION_THRESHOLD` a la probabilidad: `prediction = 1` si `probability_canceled >= umbral`.

In [4]:
resultado = predict_dataframe(df_ejemplo, model=modelo)
resultado

,prediction,probability_canceled
0,0,0.4157


In [5]:
fila = resultado.iloc[0]
proba = float(fila['probability_canceled'])
clase = int(fila['prediction'])
etiqueta = config.CLASS_LABELS_SHORT[clase]
umbral = config.DECISION_THRESHOLD

print(f'Probabilidad de cancelación: {proba:.2%}')
print(f'Umbral de decisión:          {umbral:.2%}')
print(f'Clase predicha:              {clase} -> "{etiqueta}"')
print()
if proba >= umbral:
    print(f'Como {proba:.2%} >= {umbral:.0%}, el modelo predice CANCELACIÓN.')
else:
    print(f'Como {proba:.2%} < {umbral:.0%}, el modelo predice que NO se cancela.')

Probabilidad de cancelación: 41.57%
Umbral de decisión:          50.00%
Clase predicha:              0 -> "No cancelada"

Como 41.57% < 50%, el modelo predice que NO se cancela.


## 3. Predecir un lote de reservas reales

Ahora tomamos ~10 reservas **reales** del dataset original con `load_raw_data()`. Para ser honestos, **descartamos la columna objetivo** (`is_canceled`) antes de predecir: el modelo no debe verla. La guardamos aparte para luego comparar predicción frente a realidad y "contar aciertos a ojo".

> Nota: `predict_dataframe` llama internamente a `prepare_for_inference`, así que acepta filas **en crudo** (normaliza categóricas y descarta el target si viniera). No hace falta limpiarlas a mano.

In [6]:
df_raw = load_raw_data()
muestra = df_raw.sample(n=10, random_state=config.RANDOM_STATE)

# Verdad sobre el terreno (si está disponible), para comparar después.
y_real = (
    muestra[config.TARGET_COLUMN]
    if config.TARGET_COLUMN in muestra.columns
    else None
)

# El target se descarta antes de predecir (lo hace prepare_for_inference,
# pero lo dejamos explícito para que se vea la intención).
muestra_features = muestra.drop(columns=[config.TARGET_COLUMN], errors='ignore')
print('Lote a predecir:', muestra_features.shape)

Lote a predecir: (10, 31)


In [7]:
pred_lote = predict_dataframe(muestra_features, model=modelo)

# Montamos una tabla legible: predicción + probabilidad + realidad.
comparativa = pred_lote.copy()
comparativa['etiqueta_predicha'] = comparativa['prediction'].map(
    lambda c: config.CLASS_LABELS_SHORT[int(c)]
)
if y_real is not None:
    comparativa['is_canceled_real'] = y_real.astype(int)
    comparativa['acierto'] = (
        comparativa['prediction'] == comparativa['is_canceled_real']
    ).map({True: '✅', False: '❌'})
comparativa

,prediction,probability_canceled,etiqueta_predicha,is_canceled_real,acierto
30946,0,0.0345,No cancelada,0,✅
40207,1,0.9666,Cancelada,1,✅
103708,0,0.0003,No cancelada,0,✅
85144,0,0.0077,No cancelada,0,✅
109991,0,0.4383,No cancelada,0,✅
110622,0,0.4152,No cancelada,0,✅
47790,1,0.9417,Cancelada,1,✅
44992,0,0.0005,No cancelada,0,✅
30528,0,0.0028,No cancelada,0,✅
16886,0,0.0012,No cancelada,0,✅


In [8]:
# Resumen de aciertos sobre este pequeño lote (no es una métrica formal:
# para eso está el notebook de evaluación sobre el conjunto de test).
if y_real is not None:
    n_ok = int((pred_lote['prediction'].values == y_real.astype(int).values).sum())
    print(f'Aciertos en el lote: {n_ok}/{len(pred_lote)} ({n_ok / len(pred_lote):.0%})')
else:
    print('El dataset no incluye la columna objetivo: solo se muestran predicciones.')

Aciertos en el lote: 10/10 (100%)


## 4. (Opcional) Llamar a la API

El **mismo** `best_model.pkl` se sirve por HTTP en el endpoint `/predict` de la API FastAPI. Si la tienes levantada (`make api`, o `uvicorn ml_hotel_cancellations.api.main:app --reload`), la siguiente celda le envía `config.BOOKING_EXAMPLE` y muestra la respuesta JSON.

El contrato de salida de la API (`PredictionResponse`) tiene tres campos:

- **`prediction`**: clase 0/1.
- **`label`**: etiqueta legible (`'No cancelada'` / `'Cancelada'`).
- **`probability`**: P(cancela) en [0, 1].

La celda está envuelta en `try/except`: si la API **no** está arrancada, imprime un aviso amistoso y **no rompe** la ejecución del cuaderno (importante para ejecuciones automáticas/headless).

In [9]:
import json

api_base = os.getenv('PONTIA_API_URL', 'http://localhost:8000')
url = api_base.rstrip('/') + '/predict'

try:
    import requests

    respuesta = requests.post(url, json=config.BOOKING_EXAMPLE, timeout=5)
    respuesta.raise_for_status()
    datos = respuesta.json()
    print(f'API respondió desde {url}:\n')
    print(json.dumps(datos, indent=2, ensure_ascii=False))
    print()
    print(
        f"-> prediction={datos.get('prediction')} "
        f"({datos.get('label')}), "
        f"probability={datos.get('probability')}"
    )
except Exception as exc:  # noqa: BLE001 - cualquier fallo de red no debe romper el notebook
    print(f'API no levantada en {url} (arráncala con `make api`).')
    print(f'Detalle: {type(exc).__name__}: {exc}')
    print('\nLa inferencia local de las secciones anteriores no necesita la API.')

API respondió desde http://localhost:8000/predict:

{
  "prediction": 0,
  "label": "No cancelada",
  "probability": 0.4156999886035919
}

-> prediction=0 (No cancelada), probability=0.4156999886035919


## 5. Conclusión

El mismo artefacto `models/best_model.pkl` alimenta los **tres** consumidores del proyecto:

- este **notebook** (vía `predict_dataframe`),
- la **API** FastAPI (endpoint `/predict`),
- y la **UI** Streamlit.

Esa es la idea clave: **una sola fuente de verdad para la inferencia**. El preprocesado viaja dentro del `Pipeline`, así que las predicciones son idénticas en todas partes y no hay riesgo de divergencia entre lo que se entrenó y lo que se sirve. El umbral (`config.DECISION_THRESHOLD`) y el contrato de entrada (`config.BOOKING_EXAMPLE`) también están centralizados en `config.py`.

Con esto cierra el recorrido: EDA → preprocesado → entrenamiento/comparación → evaluación → **inferencia en producción**.